### LIMPIEZA DE DATOS DE PRECIPITACIÓN - GENERACIÓN DE TABLAS DE PRECIPITACIÓN MENSUAL

In [1]:
# Dependencias

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Config

ROOT = Path.cwd().resolve().parent
GROUND = ROOT / "ground_stations"
MINIMO_DATOS = 22         # número mínimo de días con datos para que un mes sea válido

In [3]:
# Función para extraer datos de tablas en formato largo

def extract_from_long(
    path: str, 
    nombre_estación: str, 
    col_fecha: str, 
    col_p: str
):
    """
    Extrae y procesa datos de precipitación de tablas en formato largo.
    Devuelve un DataFrame con agregados mensuales, columnas: fecha, p_mensual, estacion.
    """
    
    # Función interna para leer CSV con fallback de codificación
    def leer_csv_con_fallback(p):
        try:
            return pd.read_csv(p, encoding='utf-8')
        except UnicodeDecodeError:
            print(f"{p} no se pudo leer con utf-8. Probando con latin1...")
            return pd.read_csv(p, encoding='latin1')

    # Leer archivo
    raw_data = leer_csv_con_fallback(path)
    
    # Verificar que las columnas estén bien identificadas
    if col_fecha not in raw_data.columns or col_p not in raw_data.columns:
        raise ValueError(f"Columnas '{col_fecha}' o '{col_p}' no encontradas en {path}")
    
    # Renombrar columnas de fecha y precipitación
    renamed = raw_data.rename(columns={col_fecha: 'fecha', col_p: 'p'})

   # Conversión de tipos
    renamed['fecha'] = pd.to_datetime(renamed['fecha'], errors='coerce')
    renamed['p'] = pd.to_numeric(renamed['p'], errors='coerce', downcast='float')

    # Agregaciones
    renamed['mes'] = renamed['fecha'].dt.month
    renamed['año'] = renamed['fecha'].dt.year
    
    p_mensual_group = renamed['p'].groupby([renamed['año'], renamed['mes']]).sum()
    dias = renamed['p'].groupby([renamed['año'], renamed['mes']]).count()
    p_mensual_filtrado = p_mensual_group[dias >= MINIMO_DATOS]

    # Convertir la Series de groupby() a DF
    p_mensual_df = p_mensual_filtrado.reset_index()
    
    # Agregar columna de estación (para concatenar después)
    p_mensual_df['estacion'] = nombre_estación

    # Renombrar columna precipitación
    p_mensual_df = p_mensual_df.rename(columns={'p': 'p_mensual'})
    
    # Asegurar formatos
    p_mensual_df['fecha'] = pd.to_datetime({
        'year': p_mensual_df['año'],
        'month': p_mensual_df['mes'],
        'day': 1
    })
    p_mensual_df['p_mensual'] = p_mensual_df['p_mensual'].astype('int64')

    p_mensual = p_mensual_df[['fecha', 'p_mensual', 'estacion']]
    
    return p_mensual

In [4]:
# Función para extraer datos de tablas en formato ancho

def extract_from_wide(path: str, nombre_estacion: str, col_año: str = 'año'):
    """
    Extrae datos de precipitación mensual de tablas en formato ancho (meses como columnas).
    Devuelve un DataFrame con columnas: fecha, p_mensual, estacion.
    """

    # Leer CSV (con fallback de codificación)
    def leer_csv_con_fallback(p):
        try:
            return pd.read_csv(p, encoding='utf-8')
        except UnicodeDecodeError:
            print(f"{p} no se pudo leer con utf-8. Probando con latin1...")
            return pd.read_csv(p, encoding='latin1')

    raw_data = leer_csv_con_fallback(path)

    # Renombrar columna de año si es necesario
    if col_año != 'año':
        raw_data = raw_data.rename(columns={col_año: 'año'})

    # Derretir la tabla: pasar de ancho a largo
    df_largo = raw_data.melt(id_vars='año', var_name='mes_str', value_name='p_mensual')

    # Mapear nombre de meses a números
    meses_map = {
        'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
        'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12
    }

    df_largo['mes'] = df_largo['mes_str'].map(meses_map)

    # Eliminar filas con meses no reconocidos
    df_largo = df_largo.dropna(subset=['mes'])

    # Convertir a numérico
    df_largo['p_mensual'] = pd.to_numeric(df_largo['p_mensual'], errors='coerce', downcast='float')
    df_largo['p_mensual'] = df_largo['p_mensual'].astype('int64')

    # Generar columna fecha
    df_largo['fecha'] = pd.to_datetime({
        'year': df_largo['año'].astype(int),
        'month': df_largo['mes'].astype(int),
        'day': 1
    })

    # Agregar estación
    df_largo['estacion'] = nombre_estacion

    # Salida final
    p_mensual = df_largo[['fecha', 'p_mensual', 'estacion']]

    return p_mensual

In [5]:
# Cargar y procesar Villa Gesell (largo)

path_vg = GROUND / "vg_2000_2024.csv"
nombre_vg = "villa_gesell"
col_fecha_vg = "date"
col_p_vg = "daily_p"

p_mensual_vg = extract_from_long(path_vg, nombre_vg, col_fecha_vg, col_p_vg)
p_mensual_vg

C:\Users\Usuario\AppData\Local\Temp\ipykernel_12156\1403995692.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  renamed['fecha'] = pd.to_datetime(renamed['fecha'], errors='coerce')


,fecha,p_mensual,estacion
0,2000-01-01,74,villa_gesell
1,2000-02-01,217,villa_gesell
2,2000-03-01,88,villa_gesell
3,2000-04-01,51,villa_gesell
4,2000-05-01,49,villa_gesell
...,...,...,...
266,2024-04-01,149,villa_gesell
267,2024-05-01,14,villa_gesell
268,2024-06-01,32,villa_gesell
269,2024-07-01,25,villa_gesell


In [6]:
# Cargar y procesar La Plata (largo)

path_lp_aero = GROUND / "lp_aero_1984_2024.csv"
nombre_lp_aero = 'lp_aero'
col_fecha_lp_aero = 'fecha'
col_p_lp_aero = 'prcp (mm)'

p_mensual_lp = extract_from_long(path_lp_aero, nombre_lp_aero, col_fecha_lp_aero, col_p_lp_aero)
p_mensual_lp

D:\Projects\chirps-validation\ground_stations\lp_aero_1984_2024.csv no se pudo leer con utf-8. Probando con latin1...


,fecha,p_mensual,estacion
0,1984-01-01,196,lp_aero
1,1984-02-01,259,lp_aero
2,1984-03-01,124,lp_aero
3,1984-04-01,50,lp_aero
4,1984-05-01,85,lp_aero
...,...,...,...
487,2024-08-01,111,lp_aero
488,2024-09-01,19,lp_aero
489,2024-10-01,37,lp_aero
490,2024-11-01,123,lp_aero


In [7]:
# Cargar y procesar Las Armas (largo)

path_armas = GROUND / "las_armas_2012_2025.csv"
estacion_armas = 'las_armas'
col_fecha_armas = 'Fecha'
col_p_armas = 'Precipitacion_Pluviometrica'

p_mensual_armas = extract_from_long(path_armas, estacion_armas, col_fecha_armas, col_p_armas)
p_mensual_armas

,fecha,p_mensual,estacion
0,2012-11-01,65,las_armas
1,2013-01-01,0,las_armas
2,2013-02-01,0,las_armas
3,2013-03-01,0,las_armas
4,2013-04-01,17,las_armas
...,...,...,...
101,2025-02-01,84,las_armas
102,2025-03-01,164,las_armas
103,2025-04-01,51,las_armas
104,2025-05-01,86,las_armas


In [8]:
# Cargar y procesar Lima (largo)

path_lima = GROUND / "lima_2010_2021.csv"
estacion_lima = 'lima'
col_fecha_lima = 'Fecha'
col_p_lima = 'Precipitacion_Pluviometrica'

p_mensual_lima = extract_from_long(path_lima, estacion_lima, col_fecha_lima, col_p_lima)
p_mensual_lima

,fecha,p_mensual,estacion
0,2010-09-01,107,lima
1,2010-10-01,48,lima
2,2010-11-01,52,lima
3,2010-12-01,52,lima
4,2011-01-01,114,lima
...,...,...,...
90,2019-11-01,99,lima
91,2019-12-01,95,lima
92,2021-07-01,45,lima
93,2021-08-01,24,lima


In [9]:
# Cargar y procesar San Clemente (tabla con datos sólo en días con P)

path = GROUND / "san_clemente_2003_2024.csv"
nombre_estación = 'sancle'
col_fecha = 'Fecha'
col_p = 'mm'

# Función interna para leer CSV con fallback de codificación
def leer_csv_con_fallback(p):
    try:
        return pd.read_csv(p, encoding='utf-8')
    except UnicodeDecodeError:
        print(f"{p} no se pudo leer con utf-8. Probando con latin1...")
        return pd.read_csv(p, encoding='latin1')

# Leer archivo
raw_data = leer_csv_con_fallback(path)

# Verificar que las columnas estén bien identificadas
if col_fecha not in raw_data.columns or col_p not in raw_data.columns:
    raise ValueError(f"Columnas '{col_fecha}' o '{col_p}' no encontradas en {path}")

# Renombrar columnas de fecha y precipitación
renamed = raw_data.rename(columns={col_fecha: 'fecha', col_p: 'p'})

# Convertir tipos
renamed['fecha'] = pd.to_datetime(renamed['fecha'], errors='coerce')
renamed['p'] = pd.to_numeric(renamed['p'], errors='coerce', downcast='float')


# Agregaciones
renamed['mes'] = renamed['fecha'].dt.month
renamed['año'] = renamed['fecha'].dt.year

p_mensual_group = renamed['p'].groupby([renamed['año'], renamed['mes']]).sum()

# Convertir la Series de groupby() a DF
p_mensual_df = p_mensual_group.reset_index()

# Agregar columna de estación (para concatenar después)
p_mensual_df['estacion'] = nombre_estación

# Renombrar columna precipitación
p_mensual_df = p_mensual_df.rename(columns={'p': 'p_mensual'})

# Convertir fechas a datetime
p_mensual_df['fecha'] = pd.to_datetime({
    'year': p_mensual_df['año'],
    'month': p_mensual_df['mes'],
    'day': 1
})

p_mensual_sc = p_mensual_df[['fecha', 'p_mensual', 'estacion']]
p_mensual_sc

,fecha,p_mensual,estacion
0,2003-01-01,71.0,sancle
1,2003-02-01,97.0,sancle
2,2003-03-01,48.0,sancle
3,2003-04-01,72.0,sancle
4,2003-05-01,175.0,sancle
...,...,...,...
251,2024-08-01,85.0,sancle
252,2024-09-01,20.0,sancle
253,2024-10-01,95.0,sancle
254,2024-11-01,31.0,sancle


In [10]:
# Cargar y procesar Areco (ancho)

path_areco = GROUND / "areco_1982_2022.csv"
estacion_areco = 'areco'
col_año_areco = 'Año'

p_mensual_areco = extract_from_wide(path_areco, estacion_areco, col_año_areco)
p_mensual_areco

,fecha,p_mensual,estacion
0,1982-01-01,126,areco
1,1983-01-01,139,areco
2,1984-01-01,118,areco
3,1985-01-01,65,areco
4,1986-01-01,122,areco
...,...,...,...
487,2018-12-01,189,areco
488,2019-12-01,34,areco
489,2020-12-01,43,areco
490,2021-12-01,4,areco


In [11]:
# Juntar todos los datos en un DF

p_mensual_completo = pd.concat([p_mensual_areco, p_mensual_sc, p_mensual_lima, p_mensual_lp, p_mensual_vg], ignore_index=True)[['fecha', 'p_mensual', 'estacion']].set_index('fecha')
p_mensual_completo

,p_mensual,estacion
fecha,,
1982-01-01,126.0,areco
1983-01-01,139.0,areco
1984-01-01,118.0,areco
1985-01-01,65.0,areco
1986-01-01,122.0,areco
...,...,...
2024-04-01,149.0,villa_gesell
2024-05-01,14.0,villa_gesell
2024-06-01,32.0,villa_gesell


In [12]:
p_mensual_completo.to_csv(GROUND / 'p_mensual_completo.csv', index=True)

In [13]:
p_mensual_completo['estacion'].unique()

array(['areco', 'sancle', 'lima', 'lp_aero', 'villa_gesell'], dtype=object)